In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [15, 7]
import seaborn as sns
from ipywidgets import interact,fixed

# Linear Regression with one variable

In [ ]:
rng = np.random.default_rng

### Data genereren

Hier nemen we een simpel hypothetisch voorbeeld met een dataset op basis van een wiskundige formule.

In [ ]:
x = np.arange(0,10,0.01)[np.newaxis].T
# x = np.arange(0,10,0.0025)[np.newaxis].T

In [ ]:
x.shape

In [ ]:
beta_star = np.array([0, 1, 0.1, 0.01, 5])
y_act = beta_star[0] + beta_star[1] * x + beta_star[2] * x ** 2 + beta_star[3] * x ** 3 + beta_star[4] * np.sin(x)
y = y_act + np.random.uniform(-6,6,x.shape)

In [ ]:
plt.scatter(x,y,color='k',label='all data')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

### Data opsplitsen in training en test data

We gebruiken 70% van de data voor training en 30% voor het testen.

In [ ]:
n = len(x)

ids_rand = np.random.permutation(n)

ids_train = np.sort(ids_rand[:round(0.7*n)])
ids_test = np.sort(ids_rand[round(0.7*n):])

x_train = x[ids_train]
x_test = x[ids_test]

y_train = y[ids_train]
y_test = y[ids_test]

In [ ]:
len(x_train)

### Opstellen van de feature matrix $X$

Met de `np.append` functie kun je nieuwe kollomen aan een bestaande 2D-array toevoegen.

In [ ]:
X1 = np.ones(x_train.shape)
X1 = np.append(X1,x_train,axis=1)
# np.transpose(X)
X1.shape

### Oplossen van least squares via de analytische methode

$\beta^* = (X^{\top}X)^{-1}X^{\top}y$

In [ ]:
def beta_vec(X,y):
    return np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)

In [ ]:
beta = beta_vec(X1,y_train)
display(beta)

In [ ]:
X1.shape

In [ ]:
plt.scatter(x_train,y_train,label='training data')
plt.plot(x_train,X1.dot(beta),'r',label='fit')
plt.plot(x,y_act,'g',label='onderliggende functie')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

## Bootstrappen om de modelvariantie te laten zien

In [ ]:
nboot = 1000
boot_beta0 = np.empty_like(np.zeros((nboot,1)))
boot_beta1 = np.empty_like(np.zeros((nboot,1)))

plt.scatter(x_train,y_train,label='training data')

for i in range(nboot):
    
    boot_draw = rng.choice(range(X1.shape[0]), X1.shape[0])
    beta_i = beta_vec(X1[boot_draw,:],y_train[boot_draw])
    boot_beta0[i] = beta_i[0]
    boot_beta1[i] = beta_i[1]
    plt.plot(x_train,X1.dot(beta_i),'r')

plt.plot(x_train,X1.dot(beta_i),'r',label='fitted curves')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'$n = {X1.shape[0]}$ datapunten')
plt.legend()


Modelvariantie komt ook terug in de parameters $\beta_0,\beta_1$. Dit zijn op zichzelf statistieken die hun eigen samplingdistributie hebben.

In [ ]:
plt.subplot(1, 2, 1)
sns.histplot(boot_beta0,legend=False)
plt.xlabel(r'$\beta_0$')
plt.title(f'$n = {X1.shape[0]}$ datapunten')
plt.xlim((-8,1))
plt.subplot(1, 2, 2)
sns.histplot(boot_beta1,legend=False)
plt.xlabel(r'$\beta_1$')
plt.title(f'$n = {X1.shape[0]}$ datapunten')
plt.xlim((2,3.5))

### Feature Engineering

Hier nemen we aan dat we de structuur van de data niet weten. We weten bijvoorbeeld niet dat er een sinus in de data verscholen zit. Als eerste keuze voor het verrijken van de data set kies ik ervoor om hogere orde polynomials van $x$ toe te voegen. Let op: dit is een keuze, en niet per sé de standaard manier om te starten. Het is een zelfgemaakt voorbeeld om het proces van feature engineering te laten zien.

$\left[ 1 , x , x^2 , x^3 , \ldots , x^d  \right]$


In [ ]:
X2 = np.ones(x_train.shape)
X2 = np.append(X2, x_train, axis=1)

def build_X_polyn(x,X,p):
    for i in p:
        X = np.append(X, x ** i, axis=1)
    return X

d_max = 14 # stel de maximale orde van de input feature polynomials in
X2 = build_X_polyn(x_train,X2,range(2,d_max+1))

X2.shape

I los nu de least squares op om de coefficiënten beta te bepalen voor een toenemend aantal polynomials in de matrix $X$. (Hier creëer ik één matrix `X` en selecteer een steeds groter aantal kolommen)

In [ ]:
betas = [beta_vec(X2[:,:d],y_train) for d in range(1,d_max+1)]
display(betas)

Hier plotten we de oplossing voor één geselecteerde orde.

In [ ]:
d_sel = 11

plt.scatter(x_train,y_train,label='training data')
plt.plot(x_train,X2[:,:d_sel+1].dot(betas[d_sel]),'r',label='fit')
plt.scatter(x_test,y_test,c='g',label='test data')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

Om verschillende orde snel te vergelijken, maken we een interactieve plot aan. (We gebruiken de functie `fixed` van package `ipywidgets` om de inputvariabelen die we niet willen variëren vast te zetten.)

In [ ]:
def plot_fit(x_act,y_act,xtr,ytr,xte,yte,betas,d):
    
    X = np.ones(xtr.shape)
    Xte = np.ones(xte.shape)
    if d > 0:
        for i in np.arange(1,d+1):
            X = np.append(X, xtr ** i, axis=1)
            Xte = np.append(Xte, xte ** i, axis=1)

    plt.scatter(xtr,ytr,label='training data')
    plt.plot(xtr,X.dot(betas[d]),'r',label='fit')
#     plt.scatter(xte,yte,c='k',label='test data')
    plt.plot(x_act,y_act,'g',label='onderliggende functie')
    
    plt.title(f'Orde {d} model')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()

interact(plot_fit, x_act=fixed(x), y_act=fixed(y_act), xtr=fixed(x_train), ytr=fixed(y_train), xte=fixed(x_test), yte=fixed(y_test), betas=fixed(betas), d = (0,d_max-1,1));

## Bootstrappen om de modelvariantie te laten zien

In [ ]:
d = 6

nboot = 1000
boot_beta = np.empty_like(np.zeros((d+1,nboot)))


plt.scatter(x_train,y_train,label='training data')

for i in range(nboot):
    
    boot_draw = rng.choice(range(X1.shape[0]), X1.shape[0])
    beta_i = beta_vec(X2[boot_draw,:d+1],y_train[boot_draw])
    boot_beta[:,i:i+1] = beta_i
    plt.plot(x_train,X2[:,:d+1].dot(beta_i),'r')

plt.plot(x_train,X2[:,:d+1].dot(beta_i),'r',label='fitted curves')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'$n = {X2.shape[0]}$ datapunten, model orde {d}')
plt.legend()

## Adding a sine function feature

Als we weten dat er een sinus aanwezig is in het signaal, kunnen we deze uiteraard toevoegen.

In [ ]:
X3 = np.ones(x_train.shape)
X3 = np.append(X3,x_train,axis=1)
X3 = np.append(X3,x_train**2,axis=1)
X3 = np.append(X3,x_train**3,axis=1)
X3 = np.append(X3, np.sin(x_train), axis=1)

beta_sin = beta_vec(X3,y_train)

In [ ]:
plt.scatter(x_train,y_train,label='training data')
plt.plot(x_train,X3.dot(beta_sin),'r',label='fit')
plt.scatter(x_test,y_test,c='k',label='test data')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

Als laatste werpen we een blik op de beta coëfficienten om te zien of deze in de buurt liggen van de onderliggende parameters (zie bovenin het notebook waar de vector `y` wordt opgesteld.

In [ ]:
display(beta_sin)

Alle waardes liggen wel in de orde van de onderliggende parameters, maar wijken toch ook nog behoorlijk af. Dit geeft je een idee dat het niet vanzelfsprekend is dat je machine learning parameters zo maar als vertegenwoordiging van de onderliggende werkelijkheid kunt interpreteren. Dat vergt verdere analyse.

In [ ]:
nboot = 1000
boot_beta0 = np.empty_like(np.zeros((nboot,1)))
boot_beta1 = np.empty_like(np.zeros((nboot,1)))
boot_beta2 = np.empty_like(np.zeros((nboot,1)))
boot_beta3 = np.empty_like(np.zeros((nboot,1)))
boot_beta4 = np.empty_like(np.zeros((nboot,1)))

plt.scatter(x_train,y_train,label='training data')

for i in range(nboot):
    
    boot_draw = rng.choice(range(X3.shape[0]), X3.shape[0])
    beta_i = beta_vec(X3[boot_draw,:],y_train[boot_draw])
    boot_beta0[i] = beta_i[0]
    boot_beta1[i] = beta_i[1]
    boot_beta2[i] = beta_i[2]
    boot_beta3[i] = beta_i[3]
    boot_beta4[i] = beta_i[4]
    plt.plot(x_train,X3.dot(beta_i),'r')

plt.plot(x_train,X3.dot(beta_i),'r',label='fitted curves')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'$n = {X3.shape[0]}$ datapunten')
plt.legend()


In [ ]:
plt.subplot(1, 5, 1)
sns.histplot(boot_beta0,legend=False)
plt.axvline(x=beta_star[0],c='red')
plt.xlabel(r'$\beta_0$')
plt.title(f'$n = {X3.shape[0]}$ datapunten')
# plt.xlim((-8,1))

plt.subplot(1, 5, 2)
sns.histplot(boot_beta1,legend=False)
plt.axvline(x=beta_star[1],c='red')
plt.xlabel(r'$\beta_1$')
plt.title(f'$n = {X3.shape[0]}$ datapunten')
# plt.xlim((2,3.5))

plt.subplot(1, 5, 3)
sns.histplot(boot_beta2,legend=False)
plt.axvline(x=beta_star[2],c='red')
plt.xlabel(r'$\beta_2$')
plt.title(f'$n = {X3.shape[0]}$ datapunten')

plt.subplot(1, 5, 4)
sns.histplot(boot_beta3,legend=False)
plt.axvline(x=beta_star[3],c='red')
plt.xlabel(r'$\beta_3$')
plt.title(f'$n = {X3.shape[0]}$ datapunten')

plt.subplot(1, 5, 5)
sns.histplot(boot_beta4,legend=False)
plt.axvline(x=beta_star[4],c='red')
plt.xlabel(r'$\beta_4$')
plt.title(f'$n = {X3.shape[0]}$ datapunten')